In [1]:
# Setup 
%pip install -U torch tensorboard
%pip install -U transformers datasets accelerate evaluate bitsandbytes trl peft protobuf sentencepiece lxml
#%pip install flash-attn

  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached grpcio-1.80.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached trl-1.2.0-py3-none-any.whl.metadata (11 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached lxml-6.1.0-cp313-cp313-win_amd64.whl.metadata (4.1 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.2-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-a


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# On first installation need to run this cell to fix import bug with utf-8 encoding in trl package

from pathlib import Path
import sys
import re

targets = []
for p in map(Path, sys.path):
    candidate = p / "trl" / "chat_template_utils.py"
    if candidate.exists():
        targets.append(candidate)

if not targets:
    raise FileNotFoundError("trl/chat_template_utils.py nicht gefunden.")

trl_file = targets[0]
print("Gefunden:", trl_file)

text = trl_file.read_text(encoding="utf-8")

patched = text.replace(
    '.read_text()',
    '.read_text(encoding="utf-8")'
)

if patched == text:
    print("Schon gepatcht oder keine passende Stelle gefunden.")
else:
    trl_file.write_text(patched, encoding="utf-8")
    print("Patch geschrieben.")

Gefunden: f:\VSCode\ProcessModels\daprecokb\gdpr\llm\Lib\site-packages\trl\chat_template_utils.py
Patch geschrieben.


In [5]:
# Imports
from pathlib import Path
import json
import re
import statistics
from datasets import load_dataset, DatasetDict, load_from_disk
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

dataset = "training_data_LLM_format"
model_id = "google/gemma-4-E4B-it"
tokenizer_id = "google/gemma-4-E4B-it"
output_dir = "gemma4_finetuned"
SEED = 42

In [ ]:
# Dataset loading and splitting -> using train 90% val 5% test 5% since I dont have much data

dataset = load_dataset("json", data_files=dataset, split="train").shuffle(seed=SEED)

train_temp = dataset.train_test_split(test_size=0.10, seed=SEED) # 90%train
val_test = train_temp["test"].train_test_split(test_size=0.40, seed=SEED) # remaining 50% split into 5% val and 5% test

dataset_splits = DatasetDict({
    "train": train_temp["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

dataset_splits.save_to_disk("training_split")
dataset_splits

Saving the dataset (1/1 shards): 100%|██████████| 2/2 [00:00<00:00, 267.78 examples/s]


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 45
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 3
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 2
    })
})

In [3]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)

def chat_len(messages):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return len(tokenizer(rendered)["input_ids"])

lengths = [chat_len(ex["messages"]) for ex in dataset_splits["train"]]

def percentile(values, p):
    idx = min(len(values) - 1, int(len(values) * p))
    return sorted(values)[idx]

print("count:", len(lengths))
print("min:", min(lengths))
print("p50:", statistics.median(lengths))
print("p90:", percentile(lengths, 0.90))
print("p95:", percentile(lengths, 0.95))
print("max:", max(lengths))

f:\VSCode\ProcessModels\daprecokb\gdpr\llm\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maxim\.cache\huggingface\hub\models--google--gemma-4-E4B-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


count: 45
min: 871
p50: 3158
p90: 5806
p95: 6647
max: 9459


In [ ]:
# Load model and tokenizer
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

model_kwargs = dict(
    dtype=torch_dtype,
    device_map="auto",
)

model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_quant_storage=torch_dtype,
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    **model_kwargs
)

tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)

print("Model + tokenizer loaded.")

In [ ]:
# Hyperparameters

MAX_LENGTH = 10240

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)

args = SFTConfig(
    output_dir="output_dir",
    max_length=MAX_LENGTH,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    packing = False,
    learning_rate = 2e-5,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    optim="adamw_torch_fused",
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    max_grad_norm=0.3,
    lr_scheduler_type="constant",
    report_to="tensorboard",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": True,
    },
)

In [ ]:
# Trainer
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(output_dir)

In [ ]:
del trainer
del model
torch.cuda.empty_cache()